# Kaggle 性別預測模型建立完整流程
modelXGBRf30

In [6]:
# !pip install imbalanced-learn missingpy xgboost catboost optuna sentence-transformers -q

In [7]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours
from imblearn.combine import SMOTEENN
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

## 第一步：準備資料與預防資料洩漏
讀取資料並先進行 Train/Test Split (抽出驗證集)。

In [8]:
train_path = 'data/boy or girl 2025 train_missingValue.csv'
test_path = 'data/boy or girl 2025 test no ans_missingValue.csv'

df_train_full = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

X_full = df_train_full.drop(columns=['gender'])
y_full = df_train_full['gender']

X_train, X_valid, y_train, y_valid = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)
print('Train shape:', X_train.shape)
print('Valid shape:', X_valid.shape)
print('Test shape:', df_test.shape)

Train shape: (338, 10)
Valid shape: (85, 10)
Test shape: (426, 11)


## Phase 0, 1, 2: 資料清理、異常值處理與特徵萃取

In [9]:
# IQR-based outlier detection functions
def find_outliers_iqr_with_bounds(series, iqr_multiplier=1.5, min_clip=None, max_clip=None):
    """Enhanced IQR outlier detection that returns both outlier indicators and clipping bounds."""
    valid_data = series.dropna()
    if len(valid_data) < 4:  # Need at least 4 points for meaningful IQR
        return {
            'outliers': pd.Series([False] * len(series), index=series.index),
            'lower_bound': valid_data.min() if len(valid_data) > 0 else 0,
            'upper_bound': valid_data.max() if len(valid_data) > 0 else 1,
            'clip_lower': min_clip if min_clip is not None else (valid_data.min() if len(valid_data) > 0 else 0),
            'clip_upper': max_clip if max_clip is not None else (valid_data.max() if len(valid_data) > 0 else 1),
            'outlier_values': pd.Series([], dtype=series.dtype)
        }

    Q1 = valid_data.quantile(0.25)
    Q3 = valid_data.quantile(0.75)
    IQR = Q3 - Q1

    if IQR == 0:  # All values are the same
        return {
            'outliers': pd.Series([False] * len(series), index=series.index),
            'lower_bound': Q1, 'upper_bound': Q3,
            'clip_lower': min_clip if min_clip is not None else Q1,
            'clip_upper': max_clip if max_clip is not None else Q3,
            'outlier_values': pd.Series([], dtype=series.dtype)
        }

    lower_bound = Q1 - iqr_multiplier * IQR
    upper_bound = Q3 + iqr_multiplier * IQR
    clip_lower = max(lower_bound, min_clip) if min_clip is not None else lower_bound
    clip_upper = min(upper_bound, max_clip) if max_clip is not None else upper_bound

    outliers = (series < lower_bound) | (series > upper_bound)
    outliers = outliers.fillna(False)
    outlier_values = series[outliers]

    return {
        'outliers': outliers, 'lower_bound': lower_bound, 'upper_bound': upper_bound,
        'clip_lower': clip_lower, 'clip_upper': clip_upper, 'outlier_values': outlier_values
    }

def is_repeated_or_symbol(s):
    if pd.isna(s) or s == '' or s == 'nan': return False
    if re.fullmatch(r'[^A-Za-z0-9\u4e00-\u9fff]+', s): return True
    if re.fullmatch(r'(.)\1{3,}', s): return True
    return False

def build_features(df, use_iqr=True, iqr_multiplier=1.5):
    """Enhanced feature engineering with IQR-based outlier detection for XGBRf30 model."""
    df = df.copy()

    # Phase 0: Data type validation & invalid data detection
    if 'yt' in df.columns:
        orig_yt = df['yt']
        df['yt'] = pd.to_numeric(df['yt'], errors='coerce')
        df['is_yt_invalid'] = np.where(orig_yt.notna() & df['yt'].isna(), 1, 0)

    if 'phone_os' in df.columns:
        # Note: This version uses 'apple' instead of 'ios'
        df['phone_os_clean'] = df['phone_os'].astype(str).str.strip().str.lower().replace({'iphone': 'apple'})
        valid_os = {'android', 'apple', 'windows'}
        df['is_phone_os_invalid'] = np.where(df['phone_os_clean'].isin(valid_os), 0, 1)
        df.loc[df['is_phone_os_invalid'] == 1, 'phone_os_clean'] = 'Unknown'
        df.drop(columns=['phone_os'], inplace=True, errors='ignore')

    # Phase 1: IQR-based outlier detection and clipping
    if use_iqr:
        print("使用四分位數 (IQR) 離群值偵測...")

        # Define column-specific configurations for IQR processing
        numerical_columns = {
            'weight': {'iqr_multiplier': iqr_multiplier, 'min_clip': 40, 'max_clip': 120, 'create_missing_flag': True},
            'height': {'iqr_multiplier': iqr_multiplier, 'min_clip': 140, 'max_clip': 200, 'create_missing_flag': True},
            'iq': {'iqr_multiplier': iqr_multiplier, 'min_clip': 70, 'max_clip': 160, 'create_missing_flag': False},
            'fb_friends': {'iqr_multiplier': iqr_multiplier, 'min_clip': 0, 'max_clip': 5000, 'create_missing_flag': False},
            'yt': {'iqr_multiplier': iqr_multiplier, 'min_clip': 0, 'max_clip': 24, 'create_missing_flag': False}
        }

        # Apply IQR-based processing to each numerical column
        for col_name, config in numerical_columns.items():
            if col_name not in df.columns:
                continue

            # Create missing value flag if requested
            if config.get('create_missing_flag', False):
                df[f'is_{col_name}_missing'] = df[col_name].isna().astype(int)

            # Apply IQR outlier detection
            iqr_results = find_outliers_iqr_with_bounds(
                df[col_name],
                iqr_multiplier=config['iqr_multiplier'],
                min_clip=config.get('min_clip'),
                max_clip=config.get('max_clip')
            )

            # Create outlier flag
            df[f'is_{col_name}_outlier'] = iqr_results['outliers'].astype(int)

            # Apply clipping
            df[col_name] = df[col_name].clip(
                lower=iqr_results['clip_lower'],
                upper=iqr_results['clip_upper']
            )

            # Print summary
            n_outliers = iqr_results['outliers'].sum()
            if n_outliers > 0:
                print(f"   {col_name}: 發現 {n_outliers} 個離群值。"
                      f"IQR 界線: [{iqr_results['lower_bound']:.2f}, {iqr_results['upper_bound']:.2f}]。"
                      f"截斷至: [{iqr_results['clip_lower']:.2f}, {iqr_results['clip_upper']:.2f}]")

        # Handle YouTube outliers with invalid flag combination
        if 'yt' in df.columns:
            existing_yt_outlier = df.get('is_yt_outlier', 0)
            existing_yt_invalid = df.get('is_yt_invalid', 0)
            df['is_yt_outlier'] = np.where((existing_yt_outlier == 1) | (existing_yt_invalid == 1), 1, 0)
    else:
        print("使用硬編碼閾值離群值偵測 (備用模式)...")
        # Original hardcoded approach as fallback
        if 'weight' in df.columns:
            df['is_weight_missing'] = df['weight'].isna().astype(int)
            df['is_weight_outlier'] = np.where(df['weight'].notna() & ((df['weight'] < 30) | (df['weight'] > 1000)), 1, 0)
            df['weight'] = df['weight'].clip(lower=40, upper=120)
        if 'height' in df.columns:
            df['is_height_missing'] = df['height'].isna().astype(int)
            df['is_height_outlier'] = np.where(df['height'].notna() & ((df['height'] < 130) | (df['height'] > 220)), 1, 0)
            df['height'] = df['height'].clip(lower=140, upper=200)
        if 'iq' in df.columns:
            df['is_iq_outlier'] = np.where(df['iq'].notna() & ((df['iq'] < 100) | (df['iq'] > 140)), 1, 0)
            df['iq'] = df['iq'].clip(lower=100, upper=140)
        if 'fb_friends' in df.columns:
            df['is_fb_friends_outlier'] = np.where(df['fb_friends'].notna() & ((df['fb_friends'] < 0) | (df['fb_friends'] > 2000)), 1, 0)
            df['fb_friends'] = df['fb_friends'].clip(lower=0, upper=2000)
        if 'yt' in df.columns:
            df['is_yt_outlier'] = np.where(df['yt'].notna() & ((df['yt'] < 0) | (df['yt'] > 24)), 1, df.get('is_yt_invalid', 0))
            df['yt'] = df['yt'].clip(lower=0, upper=24)

    # Star sign validation with Chinese character mapping
    if 'star_sign' in df.columns:
        # Mapping for Chinese zodiac names
        star_map = {
            '牡羊座': 'aries', '金牛座': 'taurus', '雙子座': 'gemini', '巨蟹座': 'cancer',
            '獅子座': 'leo', '處女座': 'virgo', '天秤座': 'libra', '天蠍座': 'scorpio',
            '射手座': 'sagittarius', '摩羯座': 'capricorn', '水瓶座': 'aquarius', '雙魚座': 'pisces'
        }

        # Clean up star sign data
        df['star_sign_clean'] = df['star_sign'].astype(str).str.strip()
        
        # Check if valid (whether it exists in our mapping keys)
        df['is_star_sign_invalid'] = np.where(
            df['star_sign_clean'].isin(star_map.keys()), 
            0, 1
        )
        
        # Map to English names, fill invalid ones with 'Unknown'
        df['star_sign_clean'] = df['star_sign_clean'].map(star_map).fillna('Unknown')
        
        df.drop(columns=['star_sign'], inplace=True, errors='ignore')

    # Phase 2: Text feature engineering (keeping self_intro for potential TF-IDF/BERT processing)
    if 'self_intro' in df.columns:
        df['is_intro_missing'] = df['self_intro'].isna().astype(int)
        df['intro_char_length'] = df['self_intro'].fillna('').astype(str).str.len()
        df['intro_word_count'] = df['self_intro'].fillna('').astype(str).str.split().apply(len)
        df['intro_is_all_lower'] = df['self_intro'].fillna('').astype(str).apply(lambda s: s.islower() if s else False).astype(int)
        df['intro_is_all_upper'] = df['self_intro'].fillna('').astype(str).apply(lambda s: s.isupper() if s else False).astype(int)

        df['is_intro_anomalous'] = 0
        df.loc[df['intro_char_length'] == 0, 'is_intro_anomalous'] = 1
        df.loc[df['intro_char_length'] > 500, 'is_intro_anomalous'] = 1
        df.loc[df['self_intro'].fillna('').astype(str).apply(is_repeated_or_symbol), 'is_intro_anomalous'] = 1

        # Set anomalous text to NaN, but keep the column for potential TF-IDF/BERT feature extraction
        df.loc[df['is_intro_anomalous'] == 1, 'self_intro'] = np.nan

    return df

# Apply feature engineering with IQR-based outlier detection
X_train_clean = build_features(X_train, use_iqr=True, iqr_multiplier=1.5)
X_valid_clean = build_features(X_valid, use_iqr=True, iqr_multiplier=1.5)
X_test_clean = build_features(df_test, use_iqr=True, iqr_multiplier=1.5)

print("特徵工程完成 (使用四分位數離群值偵測 - XGBRf30 版本)")

# Optional: Test on specific ID (uncomment if needed)
# test = build_features(X_train[X_train['id'] == 163], use_iqr=True, iqr_multiplier=1.5)

使用四分位數 (IQR) 離群值偵測...
   weight: 發現 9 個離群值。IQR 界線: [28.00, 100.00]。截斷至: [40.00, 100.00]
   height: 發現 11 個離群值。IQR 界線: [147.00, 195.00]。截斷至: [147.00, 195.00]
   fb_friends: 發現 22 個離群值。IQR 界線: [-550.00, 1450.00]。截斷至: [0.00, 1450.00]
   yt: 發現 45 個離群值。IQR 界線: [-12.50, 23.50]。截斷至: [0.00, 23.50]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 4 個離群值。IQR 界線: [18.12, 103.12]。截斷至: [40.00, 103.12]
   height: 發現 3 個離群值。IQR 界線: [150.19, 189.69]。截斷至: [150.19, 189.69]
   fb_friends: 發現 7 個離群值。IQR 界線: [-675.00, 1525.00]。截斷至: [0.00, 1525.00]
   yt: 發現 11 個離群值。IQR 界線: [-9.50, 18.50]。截斷至: [0.00, 18.50]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 11 個離群值。IQR 界線: [21.50, 105.50]。截斷至: [40.00, 105.50]
   height: 發現 12 個離群值。IQR 界線: [145.00, 193.00]。截斷至: [145.00, 193.00]
   fb_friends: 發現 20 個離群值。IQR 界線: [-433.62, 1235.38]。截斷至: [0.00, 1235.38]
   yt: 發現 55 個離群值。IQR 界線: [-70.00, 122.00]。截斷至: [0.00, 24.00]
特徵工程完成 (使用四分位數離群值偵測 - XGBRf30 版本)


## 前處理總攬
1. `SelfIntroEncoder`：只在 `fit(train)` 計算文字原型/PCA；`transform` 只套用。
2. `TabularPreprocessor`：只在 `fit(train)` 做補值與 one-hot 欄位定義；`transform` 對齊欄位。
3. `FullPreprocessor`：把上面兩段串起來，保證 `CV` 每個 fold 都能獨立 fit，避免 leakage。
4. 使用方式：
   - 不做 CV：用 `run_preprocess_no_cv_v2(...)`
   - 做 CV：在每個 fold 用 `preprocess_one_fold_v2(...)`

In [10]:
import copy
from dataclasses import dataclass

from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

# Default setting for class-based pipeline (kept for backward compatibility)
if 'SELF_INFO_BERT_PCA_COMPONENTS' not in globals():
    SELF_INFO_BERT_PCA_COMPONENTS = -1


@dataclass
class SelfIntroEncoderConfig:
    model_name: str = 'paraphrase-multilingual-MiniLM-L12-v2'
    pca_components: int = -1  # -1: prototype only, 0: full emb, > 0: PCA dims
    use_gender_prototype: bool = True


class SelfIntroEncoder:
    def __init__(self, config: SelfIntroEncoderConfig):
        self.config = config
        self.model = None
        self.pca = None
        self.male_centroid = None
        self.female_centroid = None
        self.emb_cols = None
        self.is_fitted = False

    @staticmethod
    def _has_intro_mask(df):
        text_non_empty = df['self_intro'].fillna('').astype(str).str.strip().ne('')
        if 'is_intro_missing' in df.columns:
            return df['is_intro_missing'].fillna(0).astype(int).eq(0) & text_non_empty
        return text_non_empty

    @staticmethod
    def _resolve_gender_labels(y):
        y_s = pd.Series(y)
        uniq = sorted(y_s.dropna().unique().tolist())
        if set([1, 2]).issubset(set(uniq)):
            return 1, 2
        if len(uniq) == 2:
            # For shifted labels (e.g., 0/1), treat the smaller label as class-1 counterpart.
            return uniq[0], uniq[1]
        raise ValueError(f'Expected binary labels, got: {uniq}')

    def _encode_text(self, df):
        text = df['self_intro'].fillna('').astype(str).tolist()
        return self.model.encode(text, convert_to_numpy=True, normalize_embeddings=True)

    def fit(self, X_df, y=None):
        if self.model is None:
            self.model = SentenceTransformer(self.config.model_name)

        emb = self._encode_text(X_df)

        if self.config.use_gender_prototype:
            if y is None:
                raise ValueError('y is required when use_gender_prototype=True')

            male_label, female_label = SelfIntroEncoder._resolve_gender_labels(y)
            y_s = pd.Series(y, index=X_df.index)
            has_intro = SelfIntroEncoder._has_intro_mask(X_df)

            male_mask = (y_s == male_label) & has_intro
            female_mask = (y_s == female_label) & has_intro
            if male_mask.sum() == 0:
                male_mask = (y_s == male_label)
            if female_mask.sum() == 0:
                female_mask = (y_s == female_label)

            self.male_centroid = emb[male_mask.to_numpy()].mean(axis=0)
            self.female_centroid = emb[female_mask.to_numpy()].mean(axis=0)

        if self.config.pca_components > 0:
            self.pca = PCA(n_components=self.config.pca_components, random_state=42)
            emb_reduced = self.pca.fit_transform(emb)
            self.emb_cols = [f'bert_feat_{i}' for i in range(emb_reduced.shape[1])]
        elif self.config.pca_components == 0:
            self.emb_cols = [f'bert_feat_{i}' for i in range(emb.shape[1])]

        self.is_fitted = True
        return self

    def transform(self, X_df):
        if not self.is_fitted:
            raise RuntimeError('SelfIntroEncoder must be fitted before transform().')

        emb = self._encode_text(X_df)
        features = []

        if self.config.use_gender_prototype:
            sim_m = emb @ self.male_centroid
            sim_f = emb @ self.female_centroid
            has_intro = SelfIntroEncoder._has_intro_mask(X_df).to_numpy()
            sim_m = np.where(has_intro, sim_m, 0.0)
            sim_f = np.where(has_intro, sim_f, 0.0)
            proto_df = pd.DataFrame({
                'self_intro_sim_to_male': sim_m,
                'self_intro_sim_to_female': sim_f,
                'self_intro_male_minus_female': sim_m - sim_f
            }, index=X_df.index)
            features.append(proto_df)

        if self.config.pca_components == 0:
            emb_df = pd.DataFrame(emb, columns=self.emb_cols, index=X_df.index)
            features.append(emb_df)
        elif self.config.pca_components > 0:
            emb_reduced = self.pca.transform(emb)
            emb_df = pd.DataFrame(emb_reduced, columns=self.emb_cols, index=X_df.index)
            features.append(emb_df)

        return pd.concat(features, axis=1) if features else pd.DataFrame(index=X_df.index)


class FeaturePipeline:
    def __init__(self, self_intro_encoder: SelfIntroEncoder = None):
        self.self_intro_encoder = self_intro_encoder
        self.is_fitted = False

    def clone(self):
        return copy.deepcopy(self)

    def _base_transform(self, X_df):
        return build_features(X_df)

    def fit(self, X_df, y=None):
        X_base = self._base_transform(X_df)
        if self.self_intro_encoder is not None:
            self.self_intro_encoder.fit(X_base, y)
        self.is_fitted = True
        return self

    def transform(self, X_df):
        if not self.is_fitted:
            raise RuntimeError('FeaturePipeline must be fitted before transform().')

        X_base = self._base_transform(X_df)
        if self.self_intro_encoder is not None:
            intro_feat = self.self_intro_encoder.transform(X_base)
            X_base = X_base.drop(columns=['self_intro'], errors='ignore')
            X_base = pd.concat([X_base, intro_feat], axis=1)
        return X_base

    def fit_transform(self, X_df, y=None):
        return self.fit(X_df, y).transform(X_df)


class TabularPreprocessor:
    def __init__(self):
        self.num_cols = None
        self.cat_cols = None
        self.cat_fill_values = {}
        self.imputer = None
        self.train_columns_ = None
        self.is_fitted = False

    def fit(self, X_df):
        X = X_df.copy()
        self.num_cols = [c for c in ['height', 'weight', 'iq', 'fb_friends', 'yt', 'sleepiness'] if c in X.columns]
        self.cat_cols = [c for c in ['phone_os_clean', 'star_sign_clean'] if c in X.columns]

        for c in self.cat_cols:
            mode = X[c].mode()[0] if len(X[c].mode()) > 0 else 'Unknown'
            self.cat_fill_values[c] = mode
            X[c] = X[c].fillna(mode)

        if len(self.num_cols) > 0:
            rf_estimator = RandomForestRegressor(n_estimators=50, random_state=42)
            self.imputer = IterativeImputer(estimator=rf_estimator, random_state=42, max_iter=10)
            X[self.num_cols] = self.imputer.fit_transform(X[self.num_cols])

        X_enc = pd.get_dummies(X, columns=self.cat_cols, dummy_na=False, dtype=int)
        self.train_columns_ = X_enc.columns.tolist()
        self.is_fitted = True
        return self

    def transform(self, X_df):
        if not self.is_fitted:
            raise RuntimeError('TabularPreprocessor must be fitted before transform().')

        X = X_df.copy()
        for c in self.cat_cols:
            fill_val = self.cat_fill_values.get(c, 'Unknown')
            if c in X.columns:
                X[c] = X[c].fillna(fill_val)

        if len(self.num_cols) > 0:
            X[self.num_cols] = self.imputer.transform(X[self.num_cols])

        X_enc = pd.get_dummies(X, columns=self.cat_cols, dummy_na=False, dtype=int)
        X_enc = X_enc.reindex(columns=self.train_columns_, fill_value=0)
        return X_enc

    def fit_transform(self, X_df):
        return self.fit(X_df).transform(X_df)


class FullPreprocessor:
    def __init__(self, feature_pipeline: FeaturePipeline, tabular_preprocessor: TabularPreprocessor):
        self.feature_pipeline = feature_pipeline
        self.tabular_preprocessor = tabular_preprocessor
        self.is_fitted = False

    def clone(self):
        return copy.deepcopy(self)

    def fit(self, X_df, y=None):
        X_feat = self.feature_pipeline.fit_transform(X_df, y)
        self.tabular_preprocessor.fit(X_feat)
        self.is_fitted = True
        return self

    def transform(self, X_df):
        if not self.is_fitted:
            raise RuntimeError('FullPreprocessor must be fitted before transform().')
        X_feat = self.feature_pipeline.transform(X_df)
        return self.tabular_preprocessor.transform(X_feat)

    def fit_transform(self, X_df, y=None):
        return self.fit(X_df, y).transform(X_df)


def make_preprocess_template(pca_components=-1, use_gender_prototype=True):
    feat_pipe = FeaturePipeline(
        self_intro_encoder=SelfIntroEncoder(
            SelfIntroEncoderConfig(
                pca_components=pca_components,
                use_gender_prototype=use_gender_prototype
            )
        )
    )
    tab_pipe = TabularPreprocessor()
    return FullPreprocessor(feat_pipe, tab_pipe)


def run_preprocess_no_cv_v2(X_train_raw, y_train_raw, X_valid_raw, X_test_raw,
                            pca_components=-1, use_gender_prototype=True):
    preprocessor = make_preprocess_template(
        pca_components=pca_components,
        use_gender_prototype=use_gender_prototype
    )
    X_train_p = preprocessor.fit_transform(X_train_raw, y_train_raw)
    X_valid_p = preprocessor.transform(X_valid_raw)
    X_test_p = preprocessor.transform(X_test_raw)
    return X_train_p, X_valid_p, X_test_p, preprocessor


def preprocess_one_fold_v2(preprocessor_template, X_tr_fold, y_tr_fold, X_va_fold):
    fold_preprocessor = preprocessor_template.clone()
    X_tr_p = fold_preprocessor.fit_transform(X_tr_fold, y_tr_fold)
    X_va_p = fold_preprocessor.transform(X_va_fold)
    return X_tr_p, X_va_p, fold_preprocessor


print('Class-based preprocessing (feature + impute + one-hot) is ready.')
print('Use run_preprocess_no_cv_v2(...) for holdout mode.')
print('Use preprocess_one_fold_v2(...) inside CV folds to avoid leakage.')

Class-based preprocessing (feature + impute + one-hot) is ready.
Use run_preprocess_no_cv_v2(...) for holdout mode.
Use preprocess_one_fold_v2(...) inside CV folds to avoid leakage.


## Phase 3: Holdout 前處理（不做 CV）
這段是 `train/valid/test` 固定切分版本：
- 只用 `train` 做 `fit`（文字編碼、補值、one-hot）
- 再把同一組轉換套到 `valid/test`
- 最後把 `id` 從模型特徵移除，但保留供 submission 使用

In [11]:
# Phase 3 (rewritten): class-based preprocessing for holdout mode
# Fit on train only, transform valid/test (no leakage to holdout split).

# Fallback when cleaned feature tables are not available yet.
if 'X_train_clean' not in globals() or 'X_valid_clean' not in globals() or 'X_test_clean' not in globals():
    X_train_clean = build_features(X_train)
    X_valid_clean = build_features(X_valid)
    X_test_clean = build_features(df_test)

# Keep id before encoding drop
train_ids_raw = X_train_clean['id'].copy() if 'id' in X_train_clean.columns else None
valid_ids_raw = X_valid_clean['id'].copy() if 'id' in X_valid_clean.columns else None
test_ids_raw = X_test_clean['id'].copy() if 'id' in X_test_clean.columns else None

X_train_enc, X_valid_enc, X_test_enc, fitted_preprocessor = run_preprocess_no_cv_v2(
    X_train_clean,
    y_train,
    X_valid_clean,
    X_test_clean,
    pca_components=SELF_INFO_BERT_PCA_COMPONENTS,
    use_gender_prototype=True
)

# Keep ids for submission, and exclude id from model features
train_ids = X_train_enc.pop('id') if 'id' in X_train_enc.columns else train_ids_raw
valid_ids = X_valid_enc.pop('id') if 'id' in X_valid_enc.columns else valid_ids_raw
test_ids = X_test_enc.pop('id') if 'id' in X_test_enc.columns else test_ids_raw

print('Class-based preprocessing completed.')
print('Feature count:', X_train_enc.shape[1])
X_train_enc.head(5)

使用四分位數 (IQR) 離群值偵測...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9815.56it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


使用四分位數 (IQR) 離群值偵測...
使用四分位數 (IQR) 離群值偵測...
使用四分位數 (IQR) 離群值偵測...
使用四分位數 (IQR) 離群值偵測...
Class-based preprocessing completed.
Feature count: 41


,height,weight,sleepiness,iq,fb_friends,yt,is_yt_invalid,is_phone_os_invalid,is_weight_missing,is_weight_outlier,...,star_sign_clean_cancer,star_sign_clean_capricorn,star_sign_clean_gemini,star_sign_clean_leo,star_sign_clean_libra,star_sign_clean_pisces,star_sign_clean_sagittarius,star_sign_clean_scorpio,star_sign_clean_taurus,star_sign_clean_virgo
145,168.00,63.0,2.00,80.0,328.0,2.0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
78,170.00,62.2,3.42,99.0,99.0,10.0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
16,156.00,47.0,3.10,130.0,400.0,3.5,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
149,168.00,49.0,1.00,87.0,1450.0,23.5,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0
293,181.08,76.0,4.00,160.0,1287.0,23.5,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 最終模型訓練與評估 

In [12]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
# 設定實驗策略
resampling_strategies = ['none', 'enn', 'smote', 'smoteenn', 'xgb_cost_sensitive']
feature_selection_strategies = [
    'all_features',
    'rf_importance_mean_threshold',
    'l1_based',
    'mean_threshold',
    'ensemble_vote_2of3',
    'top_15_rf',
    'top_20_rf',
    'top_30_rf'
 ]
base_model_name = 'xgboost'

In [13]:
print(f'實驗預計執行 {len(resampling_strategies) * len(feature_selection_strategies)} 組組合...')

# XGBoost 需要連續標籤 (0, 1)
label_offset = y_train.min()
y_train_cv = y_train - label_offset
y_valid_cv = y_valid - label_offset

# 直接使用前面處理好的資料
X_train_base = X_train_enc.copy()
X_valid_base = X_valid_enc.copy()
X_test_base = X_test_enc.copy()

# train CV 設定
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def apply_resampling(strategy, X, y):
    if strategy == 'enn':
        sampler = EditedNearestNeighbours(n_neighbors=3)
    elif strategy == 'smoteenn':
        sampler = SMOTEENN(random_state=42)
    elif strategy == 'smote':
        sampler = SMOTE(random_state=42)
    elif strategy in ['none', 'xgb_cost_sensitive']:
        return X.copy(), y.copy()
    else:
        raise ValueError(f'未知重採樣策略: {strategy}')

    X_res, y_res = sampler.fit_resample(X, y)
    return X_res, y_res

# --- 核心函數：特徵選擇 ---
def select_features(strategy, X_res, y_res, X_valid_ref, X_test_ref):
    cols = X_res.columns
    cols_arr = np.array(cols)

    def fs_rf_importance_mean_threshold():
        rf_fs = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
        rf_fs.fit(X_res, y_res)
        imp = pd.Series(rf_fs.feature_importances_, index=cols)
        threshold = imp.mean()
        selected = imp[imp >= threshold].index.tolist()
        if len(selected) == 0:
            selected = imp.sort_values(ascending=False).head(20).index.tolist()
        return selected

    def fs_l1_based():
        l1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.5, max_iter=3000, random_state=42)
        l1.fit(X_res, y_res)
        coef_abs = np.abs(l1.coef_).ravel()
        selected = cols_arr[coef_abs > 1e-8].tolist()
        if len(selected) == 0:
            top_idx = np.argsort(coef_abs)[-20:]
            selected = cols_arr[top_idx].tolist()
        return selected

    def fs_mean_threshold():
        mi = mutual_info_classif(X_res, y_res, random_state=42)
        mi_s = pd.Series(mi, index=cols)
        threshold = mi_s.mean()
        selected = mi_s[mi_s >= threshold].index.tolist()
        if len(selected) == 0:
            selected = mi_s.sort_values(ascending=False).head(20).index.tolist()
        return selected

    if strategy == 'all_features':
        selected_cols = cols.tolist()

    elif strategy == 'rf_importance_mean_threshold':
        selected_cols = fs_rf_importance_mean_threshold()

    elif strategy == 'l1_based':
        selected_cols = fs_l1_based()

    elif strategy == 'mean_threshold':
        selected_cols = fs_mean_threshold()

    elif strategy == 'ensemble_vote_2of3':
        votes = {}
        fs_sets = [
            set(fs_rf_importance_mean_threshold()),
            set(fs_l1_based()),
            set(fs_mean_threshold())
        ]
        for s in fs_sets:
            for f in s:
                votes[f] = votes.get(f, 0) + 1

        # 只保留在 3 個方法中至少被選到 2 次的特徵，並維持原欄位順序
        selected_cols = [c for c in cols if votes.get(c, 0) >= 2]

        # 極端情況 fallback，避免沒有特徵可訓練
        if len(selected_cols) == 0:
            selected_cols = fs_rf_importance_mean_threshold()

    elif strategy in ['top_15_rf', 'top_20_rf', 'top_30_rf']:
        k = 15 if strategy == 'top_15_rf' else (20 if strategy == 'top_20_rf' else 30)
        rf_fs = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
        rf_fs.fit(X_res, y_res)
        imp = pd.Series(rf_fs.feature_importances_, index=cols)
        selected_cols = imp.sort_values(ascending=False).head(k).index.tolist()

    else:
        raise ValueError(f'未知特徵選擇策略: {strategy}')

    # 確保順序與去重
    selected_cols = list(dict.fromkeys(selected_cols))
    print(f"   [{strategy}] 選出 {len(selected_cols)} 個 {selected_cols}特徵。")

    X_res_sel = X_res[selected_cols]
    X_valid_sel = X_valid_ref[selected_cols]
    X_test_sel = X_test_ref[selected_cols]

    return selected_cols, X_res_sel, X_valid_sel, X_test_sel

print('函數定義完成，準備執行 Leakage-free CV...')


實驗預計執行 40 組組合...
函數定義完成，準備執行 Leakage-free CV...


## No-CV 網格實驗（與 CV 同策略）
這段會在固定 train/valid 切分下，跑完整的 resampling 與 feature selection 組合，
並輸出排名表與每組使用的特徵清單。

In [14]:
# No-CV: run all strategy combinations on fixed train/valid split
required_vars_nocv = [
    'resampling_strategies', 'feature_selection_strategies',
    'apply_resampling', 'select_features',
    'X_train_base', 'X_valid_base', 'X_test_base',
    'y_train_cv', 'y_valid_cv', 'y_valid', 'label_offset'
]
missing_nocv = [v for v in required_vars_nocv if v not in globals()]

if len(missing_nocv) > 0:
    print('請先執行前面的前處理與函數初始化 cell，缺少變數：', missing_nocv)
else:
    print('No-CV 模式：固定 train/valid，跑完整策略組合。')

    def class_accuracy(y_true, y_pred, cls):
        mask = (y_true == cls)
        if mask.sum() == 0:
            return np.nan
        return (y_pred[mask] == cls).mean()

    no_cv_rows = []
    no_cv_artifacts = []

    for rs in resampling_strategies:
        print(f"\n[No-CV][Resampling] {rs}")
        X_res, y_res = apply_resampling(rs, X_train_base, y_train_cv)

        for fs in feature_selection_strategies:
            selected_cols, X_res_sel, X_valid_sel, X_test_sel = select_features(
                fs, X_res, y_res, X_valid_base, X_test_base
            )

            print(f"  > rs={rs}, fs={fs}, n_features={len(selected_cols)}")
            print(f"    selected_features={selected_cols}")

            model_params = dict(
                objective='binary:logistic',
                eval_metric='logloss',
                learning_rate=0.05,
                n_estimators=300,
                max_depth=3,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=5,
                random_state=42
            )

            if rs == 'xgb_cost_sensitive':
                class_counts = pd.Series(y_res).value_counts()
                neg_count = class_counts.get(0, 0)
                pos_count = class_counts.get(1, 0)
                if pos_count > 0 and neg_count > 0:
                    model_params['scale_pos_weight'] = float(neg_count) / float(pos_count)

            model = XGBClassifier(**model_params)
            model.fit(X_res_sel, y_res)

            pred_valid_cv = model.predict(X_valid_sel)
            pred_valid = pred_valid_cv + label_offset

            valid_overall_acc = (pred_valid == y_valid.values).mean()
            valid_boy_acc = class_accuracy(y_valid.values, pred_valid, 1)
            valid_girl_acc = class_accuracy(y_valid.values, pred_valid, 2)
            valid_f1_macro = f1_score(y_valid_cv, pred_valid_cv, average='macro')

            no_cv_rows.append({
                'resampling': rs,
                'feature_selection': fs,
                'n_features': len(selected_cols),
                'valid_overall_acc': valid_overall_acc,
                'valid_boy_acc': valid_boy_acc,
                'valid_girl_acc': valid_girl_acc,
                'valid_f1_macro': valid_f1_macro
            })

            no_cv_artifacts.append({
                'model': model,
                'resampling': rs,
                'feature_selection': fs,
                'selected_cols': selected_cols,
                'X_test_sel': X_test_sel,
                'n_features': len(selected_cols),
                'valid_overall_acc': valid_overall_acc,
                'valid_boy_acc': valid_boy_acc,
                'valid_girl_acc': valid_girl_acc,
                'valid_f1_macro': valid_f1_macro
            })

            print(
                f"    valid_overall_acc={valid_overall_acc:.6f}, "
                f"valid_boy_acc={valid_boy_acc:.6f}, "
                f"valid_girl_acc={valid_girl_acc:.6f}, "
                f"valid_f1_macro={valid_f1_macro:.6f}"
            )

    no_cv_df = pd.DataFrame(no_cv_rows).sort_values('valid_f1_macro', ascending=False).reset_index(drop=True)
    no_cv_df.insert(0, 'rank', range(1, len(no_cv_df) + 1))

    print('\nNo-CV 策略排名結果：')
    display(no_cv_df[['rank', 'resampling', 'feature_selection', 'n_features',
                      'valid_overall_acc', 'valid_boy_acc', 'valid_girl_acc', 'valid_f1_macro']])

    # Keep top artifacts for optional no-cv submission usage
    top3_artifacts_nocv = sorted(no_cv_artifacts, key=lambda x: x['valid_f1_macro'], reverse=True)[:3]
    print('\nNo-CV Top1:')
    if len(top3_artifacts_nocv) > 0:
        print({
            'rank': 1,
            'resampling': top3_artifacts_nocv[0]['resampling'],
            'feature_selection': top3_artifacts_nocv[0]['feature_selection'],
            'n_features': top3_artifacts_nocv[0]['n_features'],
            'valid_overall_acc': top3_artifacts_nocv[0]['valid_overall_acc'],
            'valid_boy_acc': top3_artifacts_nocv[0]['valid_boy_acc'],
            'valid_girl_acc': top3_artifacts_nocv[0]['valid_girl_acc'],
            'valid_f1_macro': top3_artifacts_nocv[0]['valid_f1_macro']
        })

No-CV 模式：固定 train/valid，跑完整策略組合。

[No-CV][Resampling] none
   [all_features] 選出 41 個 ['height', 'weight', 'sleepiness', 'iq', 'fb_friends', 'yt', 'is_yt_invalid', 'is_phone_os_invalid', 'is_weight_missing', 'is_weight_outlier', 'is_height_missing', 'is_height_outlier', 'is_iq_outlier', 'is_fb_friends_outlier', 'is_yt_outlier', 'is_star_sign_invalid', 'is_intro_missing', 'intro_char_length', 'intro_word_count', 'intro_is_all_lower', 'intro_is_all_upper', 'is_intro_anomalous', 'self_intro_sim_to_male', 'self_intro_sim_to_female', 'self_intro_male_minus_female', 'phone_os_clean_Unknown', 'phone_os_clean_android', 'phone_os_clean_apple', 'star_sign_clean_Unknown', 'star_sign_clean_aquarius', 'star_sign_clean_aries', 'star_sign_clean_cancer', 'star_sign_clean_capricorn', 'star_sign_clean_gemini', 'star_sign_clean_leo', 'star_sign_clean_libra', 'star_sign_clean_pisces', 'star_sign_clean_sagittarius', 'star_sign_clean_scorpio', 'star_sign_clean_taurus', 'star_sign_clean_virgo']特徵。
  > rs=none

,rank,resampling,feature_selection,n_features,valid_overall_acc,valid_boy_acc,valid_girl_acc,valid_f1_macro
0,1,none,l1_based,12,0.917647,0.936508,0.863636,0.894222
1,2,none,top_30_rf,30,0.917647,0.936508,0.863636,0.894222
2,3,xgb_cost_sensitive,all_features,41,0.905882,0.904762,0.909091,0.883880
3,4,smote,top_15_rf,15,0.905882,0.920635,0.863636,0.880785
4,5,smote,top_20_rf,20,0.905882,0.920635,0.863636,0.880785
5,6,smote,top_30_rf,30,0.905882,0.920635,0.863636,0.880785
6,7,smote,all_features,41,0.905882,0.920635,0.863636,0.880785
7,8,none,rf_importance_mean_threshold,10,0.905882,0.936508,0.818182,0.877345
8,9,none,all_features,41,0.905882,0.936508,0.818182,0.877345
9,10,none,top_20_rf,20,0.905882,0.936508,0.818182,0.877345



No-CV Top1:
{'rank': 1, 'resampling': 'none', 'feature_selection': 'l1_based', 'n_features': 12, 'valid_overall_acc': np.float64(0.9176470588235294), 'valid_boy_acc': np.float64(0.9365079365079365), 'valid_girl_acc': np.float64(0.8636363636363636), 'valid_f1_macro': 0.8942222222222223}


## Leakage-free CV 重點說明
這段 CV 每一個 fold 都會做以下流程：
1. 用 fold-train `fit` 前處理（含文字特徵、補值、one-hot）。
2. fold-valid 只 `transform`，不參與任何 `fit`。
3. 只對 fold-train 做重採樣（SMOTE/ENN/SMOTEENN）。
4. 特徵選擇也只在 fold-train 做，避免把 fold-valid 資訊帶回模型。

In [15]:
# === Leakage-free CV (rewritten): preprocessing + resampling + feature selection all inside fold ===
required_vars_safe = [
    'resampling_strategies', 'feature_selection_strategies',
    'apply_resampling', 'select_features',
    'X_train', 'X_valid', 'df_test',
    'y_train', 'y_valid', 'cv3'
]
missing_safe = [v for v in required_vars_safe if v not in globals()]

if len(missing_safe) > 0:
    print('請先執行前一個「函數定義初始化」cell，缺少變數：', missing_safe)
else:
    # Ensure cleaned base tables exist
    if 'X_train_clean' not in globals() or 'X_valid_clean' not in globals() or 'X_test_clean' not in globals():
        X_train_clean = build_features(X_train)
        X_valid_clean = build_features(X_valid)
        X_test_clean = build_features(df_test)

    label_offset = y_train.min()
    y_train_cv = y_train - label_offset
    y_valid_cv = y_valid - label_offset

    preprocess_template = make_preprocess_template(
        pca_components=SELF_INFO_BERT_PCA_COMPONENTS,
        use_gender_prototype=True
    )

    # Holdout preprocessing for valid/test comparison (fit on full train only)
    X_train_base, X_valid_base, X_test_base, holdout_preprocessor = run_preprocess_no_cv_v2(
        X_train_clean,
        y_train,
        X_valid_clean,
        X_test_clean,
        pca_components=SELF_INFO_BERT_PCA_COMPONENTS,
        use_gender_prototype=True
    )

    # Drop id from model features but keep ids if needed
    _train_ids = X_train_base.pop('id') if 'id' in X_train_base.columns else None
    _valid_ids = X_valid_base.pop('id') if 'id' in X_valid_base.columns else None
    _test_ids = X_test_base.pop('id') if 'id' in X_test_base.columns else None

    print('Leakage-free CV 模式啟動：每個 fold 內各自前處理、重採樣、選特徵、訓練。\n')

    def _select_feature_columns_fold(strategy, X_res_fold, y_res_fold):
        cols = X_res_fold.columns
        cols_arr = np.array(cols)

        def fs_rf_importance_mean_threshold_fold():
            rf_fs = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
            rf_fs.fit(X_res_fold, y_res_fold)
            imp = pd.Series(rf_fs.feature_importances_, index=cols)
            threshold = imp.mean()
            selected = imp[imp >= threshold].index.tolist()
            if len(selected) == 0:
                selected = imp.sort_values(ascending=False).head(20).index.tolist()
            return selected

        def fs_l1_based_fold():
            l1 = LogisticRegression(
                penalty='l1', solver='liblinear', C=0.5, max_iter=3000, random_state=42
            )
            l1.fit(X_res_fold, y_res_fold)
            coef_abs = np.abs(l1.coef_).ravel()
            selected = cols_arr[coef_abs > 1e-8].tolist()
            if len(selected) == 0:
                top_idx = np.argsort(coef_abs)[-20:]
                selected = cols_arr[top_idx].tolist()
            return selected

        def fs_mean_threshold_fold():
            mi = mutual_info_classif(X_res_fold, y_res_fold, random_state=42)
            mi_s = pd.Series(mi, index=cols)
            threshold = mi_s.mean()
            selected = mi_s[mi_s >= threshold].index.tolist()
            if len(selected) == 0:
                selected = mi_s.sort_values(ascending=False).head(20).index.tolist()
            return selected

        if strategy == 'all_features':
            selected_cols_fold = cols.tolist()
        elif strategy == 'rf_importance_mean_threshold':
            selected_cols_fold = fs_rf_importance_mean_threshold_fold()
        elif strategy == 'l1_based':
            selected_cols_fold = fs_l1_based_fold()
        elif strategy == 'mean_threshold':
            selected_cols_fold = fs_mean_threshold_fold()
        elif strategy == 'ensemble_vote_2of3':
            votes = {}
            fs_sets = [
                set(fs_rf_importance_mean_threshold_fold()),
                set(fs_l1_based_fold()),
                set(fs_mean_threshold_fold())
            ]
            for s in fs_sets:
                for f in s:
                    votes[f] = votes.get(f, 0) + 1
            selected_cols_fold = [c for c in cols if votes.get(c, 0) >= 2]
            if len(selected_cols_fold) == 0:
                selected_cols_fold = fs_rf_importance_mean_threshold_fold()
        elif strategy in ['top_15_rf', 'top_20_rf', 'top_30_rf']:
            k = 15 if strategy == 'top_15_rf' else (20 if strategy == 'top_20_rf' else 30)
            rf_fs = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
            rf_fs.fit(X_res_fold, y_res_fold)
            imp = pd.Series(rf_fs.feature_importances_, index=cols)
            selected_cols_fold = imp.sort_values(ascending=False).head(k).index.tolist()
        else:
            raise ValueError(f'未知特徵選擇策略: {strategy}')

        selected_cols_fold = list(dict.fromkeys(selected_cols_fold))
        return selected_cols_fold

    def leakage_free_cv_score(rs, fs, X_raw, y_raw, cv_splitter, preproc_template):
        X_raw = X_raw.reset_index(drop=True)
        y_raw = pd.Series(y_raw).reset_index(drop=True)
        fold_scores = []

        base_params = dict(
            objective='binary:logistic',
            eval_metric='logloss',
            learning_rate=0.05,
            n_estimators=300,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=5,
            random_state=42
        )

        X_dummy = np.zeros((len(y_raw), 1))
        for _, (tr_idx, va_idx) in enumerate(cv_splitter.split(X_dummy, y_raw), start=1):
            X_tr_fold_raw = X_raw.iloc[tr_idx].copy()
            y_tr_fold_raw = y_raw.iloc[tr_idx].copy()
            X_va_fold_raw = X_raw.iloc[va_idx].copy()
            y_va_fold_raw = y_raw.iloc[va_idx].copy()

            # Fold-safe preprocessing: fit on fold-train only
            X_tr_fold, X_va_fold, _ = preprocess_one_fold_v2(
                preproc_template,
                X_tr_fold_raw,
                y_tr_fold_raw,
                X_va_fold_raw
            )

            # Drop id for model
            if 'id' in X_tr_fold.columns:
                X_tr_fold = X_tr_fold.drop(columns=['id'])
            if 'id' in X_va_fold.columns:
                X_va_fold = X_va_fold.drop(columns=['id'])

            y_tr_fold_cv = y_tr_fold_raw - label_offset
            y_va_fold_cv = y_va_fold_raw - label_offset

            X_tr_res, y_tr_res = apply_resampling(rs, X_tr_fold, y_tr_fold_cv)
            selected_cols_fold = _select_feature_columns_fold(fs, X_tr_res, y_tr_res)

            model_params = dict(base_params)
            if rs == 'xgb_cost_sensitive':
                class_counts = pd.Series(y_tr_res).value_counts()
                neg_count = class_counts.get(0, 0)
                pos_count = class_counts.get(1, 0)
                if pos_count > 0 and neg_count > 0:
                    model_params['scale_pos_weight'] = float(neg_count) / float(pos_count)

            model_fold = XGBClassifier(**model_params)
            model_fold.fit(X_tr_res[selected_cols_fold], y_tr_res)
            pred_va = model_fold.predict(X_va_fold[selected_cols_fold])
            fold_f1 = f1_score(y_va_fold_cv, pred_va, average='macro')
            fold_scores.append(fold_f1)

        return float(np.mean(fold_scores))

    print(f'Leakage-free 版本預計執行 {len(resampling_strategies) * len(feature_selection_strategies)} 組組合...')
    experiment_rows = []
    all_artifacts = []
    best_artifact = None
    best_score = -1
    resampling_sample_counts = {}

    for rs in resampling_strategies:
        print(f"\n[Resampling] 正在執行 {rs}...")
        X_res, y_res = apply_resampling(rs, X_train_base, y_train_cv)
        resampled_n = int(X_res.shape[0])
        resampling_sample_counts[rs] = resampled_n
        print(f"   -> 重採樣後訓練樣本數: {resampled_n}")

        for fs in feature_selection_strategies:
            print(f"  > [Feature Selection] {fs}", end='... ')
            selected_cols, X_res_sel, X_valid_sel, X_test_sel = select_features(
                fs, X_res, y_res, X_valid_base, X_test_base
            )

            model_params = dict(
                objective='binary:logistic',
                eval_metric='logloss',
                learning_rate=0.05,
                n_estimators=300,
                max_depth=3,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=5,
                random_state=42
            )

            if rs == 'xgb_cost_sensitive':
                class_counts = pd.Series(y_res).value_counts()
                neg_count = class_counts.get(0, 0)
                pos_count = class_counts.get(1, 0)
                if pos_count > 0 and neg_count > 0:
                    model_params['scale_pos_weight'] = float(neg_count) / float(pos_count)
                    print(f"(scale_pos_weight={model_params['scale_pos_weight']:.4f})", end=' ')

            train_cv3_mean = leakage_free_cv_score(
                rs,
                fs,
                X_train,
                y_train,
                cv3,
                preprocess_template
            )

            model = XGBClassifier(**model_params)
            model.fit(X_res_sel, y_res)

            valid_pred = model.predict(X_valid_sel)
            f1 = f1_score(y_valid_cv, valid_pred, average='macro')

            print(f"完成 (LeakFree CV3 F1: {train_cv3_mean:.4f}, Valid F1: {f1:.4f}, Features: {len(selected_cols)})")

            row = {
                'resampling': rs,
                'feature_selection': fs,
                'n_samples': resampled_n,
                'n_features': len(selected_cols),
                'train_cv3_f1_macro': train_cv3_mean,
                'valid_f1_macro': f1
            }
            experiment_rows.append(row)

            artifact = {
                'model': model,
                'resampling': rs,
                'feature_selection': fs,
                'selected_cols': selected_cols,
                'X_test_sel': X_test_sel,
                'n_samples': resampled_n,
                'train_cv3_f1_macro': train_cv3_mean,
                'valid_f1_macro': f1,
                'n_features': len(selected_cols)
            }
            all_artifacts.append(artifact)

            if f1 > best_score:
                best_score = f1
                best_artifact = artifact

    sampling_summary_df = pd.DataFrame([
        {'resampling': k, 'n_train_samples': v}
        for k, v in resampling_sample_counts.items()
    ]).sort_values('resampling').reset_index(drop=True)

    print('\n' + '=' * 30)
    print('Leakage-free 重採樣策略樣本數摘要')
    print('=' * 30)
    print(sampling_summary_df)

    experiment_df = pd.DataFrame(experiment_rows).sort_values('valid_f1_macro', ascending=False).reset_index(drop=True)
    print('\n' + '=' * 30)
    print('Leakage-free 5x8 網格實驗結果')
    print('=' * 30)
    print(experiment_df)

    top3_artifacts = sorted(all_artifacts, key=lambda x: x['valid_f1_macro'], reverse=True)[:3]
    top3_df = pd.DataFrame([
        {
            'rank': i + 1,
            'resampling': art['resampling'],
            'feature_selection': art['feature_selection'],
            'n_samples': art['n_samples'],
            'n_features': art['n_features'],
            'train_cv3_f1_macro': art['train_cv3_f1_macro'],
            'valid_f1_macro': art['valid_f1_macro']
        }
        for i, art in enumerate(top3_artifacts)
    ])

    print('\n' + '=' * 30)
    print('Leakage-free Top 3 組合')
    print('=' * 30)
    print(top3_df)

    final_model = top3_artifacts[0]['model']
    X_test_best = top3_artifacts[0]['X_test_sel']
    best_selected_features = top3_artifacts[0]['selected_cols']

使用四分位數 (IQR) 離群值偵測...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10664.76it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


使用四分位數 (IQR) 離群值偵測...
使用四分位數 (IQR) 離群值偵測...
使用四分位數 (IQR) 離群值偵測...
使用四分位數 (IQR) 離群值偵測...
Leakage-free CV 模式啟動：每個 fold 內各自前處理、重採樣、選特徵、訓練。

Leakage-free 版本預計執行 40 組組合...

[Resampling] 正在執行 none...
   -> 重採樣後訓練樣本數: 338
  > [Feature Selection] all_features...    [all_features] 選出 41 個 ['height', 'weight', 'sleepiness', 'iq', 'fb_friends', 'yt', 'is_yt_invalid', 'is_phone_os_invalid', 'is_weight_missing', 'is_weight_outlier', 'is_height_missing', 'is_height_outlier', 'is_iq_outlier', 'is_fb_friends_outlier', 'is_yt_outlier', 'is_star_sign_invalid', 'is_intro_missing', 'intro_char_length', 'intro_word_count', 'intro_is_all_lower', 'intro_is_all_upper', 'is_intro_anomalous', 'self_intro_sim_to_male', 'self_intro_sim_to_female', 'self_intro_male_minus_female', 'phone_os_clean_Unknown', 'phone_os_clean_android', 'phone_os_clean_apple', 'star_sign_clean_Unknown', 'star_sign_clean_aquarius', 'star_sign_clean_aries', 'star_sign_clean_cancer', 'star_sign_clean_capricorn', 'star_sign_clean_gemini', '

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10891.03it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


使用四分位數 (IQR) 離群值偵測...
   weight: 發現 8 個離群值。IQR 界線: [32.00, 96.00]。截斷至: [40.00, 96.00]
   height: 發現 8 個離群值。IQR 界線: [147.00, 195.00]。截斷至: [147.00, 195.00]
   fb_friends: 發現 15 個離群值。IQR 界線: [-550.00, 1450.00]。截斷至: [0.00, 1450.00]
   yt: 發現 30 個離群值。IQR 界線: [-12.50, 23.50]。截斷至: [0.00, 23.50]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 8 個離群值。IQR 界線: [32.00, 96.00]。截斷至: [40.00, 96.00]
   height: 發現 8 個離群值。IQR 界線: [147.00, 195.00]。截斷至: [147.00, 195.00]
   fb_friends: 發現 15 個離群值。IQR 界線: [-550.00, 1450.00]。截斷至: [0.00, 1450.00]
   yt: 發現 30 個離群值。IQR 界線: [-12.50, 23.50]。截斷至: [0.00, 23.50]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 3 個離群值。IQR 界線: [25.00, 105.00]。截斷至: [40.00, 105.00]
   height: 發現 3 個離群值。IQR 界線: [147.38, 194.38]。截斷至: [147.38, 194.38]
   fb_friends: 發現 7 個離群值。IQR 界線: [-425.00, 1375.00]。截斷至: [0.00, 1375.00]
   yt: 發現 15 個離群值。IQR 界線: [-13.68, 24.21]。截斷至: [0.00, 24.00]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 9 個離群值。IQR 界線: [26.50, 102.50]。截斷至: [40.00, 102.50]
   height: 發現 8 個離群值。IQR 界線: [147.00, 195.00

...

使用四分位數 (IQR) 離群值偵測...
   weight: 發現 8 個離群值。IQR 界線: [32.00, 96.00]。截斷至: [40.00, 96.00]
   height: 發現 8 個離群值。IQR 界線: [147.00, 195.00]。截斷至: [147.00, 195.00]
   fb_friends: 發現 15 個離群值。IQR 界線: [-550.00, 1450.00]。截斷至: [0.00, 1450.00]
   yt: 發現 30 個離群值。IQR 界線: [-12.50, 23.50]。截斷至: [0.00, 23.50]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 8 個離群值。IQR 界線: [32.00, 96.00]。截斷至: [40.00, 96.00]
   height: 發現 8 個離群值。IQR 界線: [147.00, 195.00]。截斷至: [147.00, 195.00]
   fb_friends: 發現 15 個離群值。IQR 界線: [-550.00, 1450.00]。截斷至: [0.00, 1450.00]
   yt: 發現 30 個離群值。IQR 界線: [-12.50, 23.50]。截斷至: [0.00, 23.50]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 3 個離群值。IQR 界線: [25.00, 105.00]。截斷至: [40.00, 105.00]
   height: 發現 3 個離群值。IQR 界線: [147.38, 194.38]。截斷至: [147.38, 194.38]
   fb_friends: 發現 7 個離群值。IQR 界線: [-425.00, 1375.00]。截斷至: [0.00, 1375.00]
   yt: 發現 15 個離群值。IQR 界線: [-13.68, 24.21]。截斷至: [0.00, 24.00]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 9 個離群值。IQR 界線: [26.50, 102.50]。截斷至: [40.00, 102.50]
   height: 發現 8 個離群值。IQR 界線: [147.00, 195.00

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11087.49it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


使用四分位數 (IQR) 離群值偵測...
   weight: 發現 9 個離群值。IQR 界線: [26.50, 102.50]。截斷至: [40.00, 102.50]
   height: 發現 8 個離群值。IQR 界線: [147.00, 195.00]。截斷至: [147.00, 195.00]
   fb_friends: 發現 15 個離群值。IQR 界線: [-550.00, 1450.00]。截斷至: [0.00, 1450.00]
   yt: 發現 34 個離群值。IQR 界線: [-15.50, 28.50]。截斷至: [0.00, 24.00]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 9 個離群值。IQR 界線: [26.50, 102.50]。截斷至: [40.00, 102.50]
   height: 發現 8 個離群值。IQR 界線: [147.00, 195.00]。截斷至: [147.00, 195.00]
   fb_friends: 發現 15 個離群值。IQR 界線: [-550.00, 1450.00]。截斷至: [0.00, 1450.00]
   yt: 發現 34 個離群值。IQR 界線: [-15.50, 28.50]。截斷至: [0.00, 24.00]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 2 個離群值。IQR 界線: [33.25, 95.25]。截斷至: [40.00, 95.25]
   height: 發現 3 個離群值。IQR 界線: [148.50, 192.50]。截斷至: [148.50, 192.50]
   fb_friends: 發現 7 個離群值。IQR 界線: [-558.25, 1463.75]。截斷至: [0.00, 1463.75]
   yt: 發現 16 個離群值。IQR 界線: [-5.00, 11.00]。截斷至: [0.00, 11.00]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 3 個離群值。IQR 界線: [28.00, 100.00]。截斷至: [40.00, 100.00]
   height: 發現 6 個離群值。IQR 界線: [148.50, 192.5

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10209.74it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


使用四分位數 (IQR) 離群值偵測...
   weight: 發現 3 個離群值。IQR 界線: [28.00, 100.00]。截斷至: [40.00, 100.00]
   height: 發現 6 個離群值。IQR 界線: [148.50, 192.50]。截斷至: [148.50, 192.50]
   fb_friends: 發現 14 個離群值。IQR 界線: [-550.00, 1450.00]。截斷至: [0.00, 1450.00]
   yt: 發現 31 個離群值。IQR 界線: [-10.30, 19.30]。截斷至: [0.00, 19.30]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 3 個離群值。IQR 界線: [28.00, 100.00]。截斷至: [40.00, 100.00]
   height: 發現 6 個離群值。IQR 界線: [148.50, 192.50]。截斷至: [148.50, 192.50]
   fb_friends: 發現 14 個離群值。IQR 界線: [-550.00, 1450.00]。截斷至: [0.00, 1450.00]
   yt: 發現 31 個離群值。IQR 界線: [-10.30, 19.30]。截斷至: [0.00, 19.30]
使用四分位數 (IQR) 離群值偵測...
   weight: 發現 6 個離群值。IQR 界線: [29.50, 97.50]。截斷至: [40.00, 97.50]
   height: 發現 5 個離群值。IQR 界線: [145.50, 197.50]。截斷至: [145.50, 197.50]
   fb_friends: 發現 12 個離群值。IQR 界線: [-488.50, 1267.50]。截斷至: [0.00, 1267.50]
   yt: 發現 19 個離群值。IQR 界線: [-20.87, 37.45]。截斷至: [0.00, 24.00]
完成 (LeakFree CV3 F1: 0.7931, Valid F1: 0.8710, Features: 30)

Leakage-free 重採樣策略樣本數摘要
           resampling  n_train_samples
0   

In [16]:
# 所有組合詳細性能對比（支援 CV / No-CV）
def class_accuracy(y_true, y_pred, cls):
    mask = (y_true == cls)
    if mask.sum() == 0:
        return np.nan
    return (y_pred[mask] == cls).mean()


def build_detail_table(artifacts, X_valid_ref, y_valid_ref, label_offset_ref, title):
    rows = []

    for rank, art in enumerate(artifacts, start=1):
        X_valid_sel = X_valid_ref[art['selected_cols']]
        pred_valid_cv = art['model'].predict(X_valid_sel)
        pred_valid = pred_valid_cv + label_offset_ref

        boy_acc = class_accuracy(y_valid_ref.values, pred_valid, 1)
        girl_acc = class_accuracy(y_valid_ref.values, pred_valid, 2)
        overall_acc = (pred_valid == y_valid_ref.values).mean()

        rows.append({
            'rank': rank,
            'resampling': art['resampling'],
            'feature_selection': art['feature_selection'],
            'n_features': art['n_features'],
            'train_cv3_f1_macro': art.get('train_cv3_f1_macro', np.nan),
            'valid_overall_acc': overall_acc,
            'valid_boy_acc': boy_acc,
            'valid_girl_acc': girl_acc,
            'valid_f1_macro': art['valid_f1_macro']
        })

    detail_df = pd.DataFrame(rows)
    detail_df = detail_df.sort_values('valid_f1_macro', ascending=False).reset_index(drop=True)
    detail_df['rank'] = range(1, len(detail_df) + 1)

    print('\n' + '=' * 80)
    print(title)
    print('=' * 80)
    display(detail_df)


required_common = ['X_valid_base', 'label_offset', 'y_valid']
missing_common = [v for v in required_common if v not in globals()]

if len(missing_common) > 0:
    print('請先執行前面的前處理與訓練 cell，缺少變數：', missing_common)
else:
    # CV 詳細表
    if 'all_artifacts' in globals() and len(all_artifacts) > 0:
        build_detail_table(
            artifacts=all_artifacts,
            X_valid_ref=X_valid_base,
            y_valid_ref=y_valid,
            label_offset_ref=label_offset,
            title='所有 CV 組合詳細性能對比 (按 valid_f1_macro 由大到小排序)'
        )
    else:
        print('CV 詳細表略過：找不到 all_artifacts。')

    # No-CV 詳細表
    if 'no_cv_artifacts' in globals() and len(no_cv_artifacts) > 0:
        build_detail_table(
            artifacts=no_cv_artifacts,
            X_valid_ref=X_valid_base,
            y_valid_ref=y_valid,
            label_offset_ref=label_offset,
            title='所有 No-CV 組合詳細性能對比 (按 valid_f1_macro 由大到小排序)'
        )
    else:
        print('No-CV 詳細表略過：找不到 no_cv_artifacts。')


所有 CV 組合詳細性能對比 (按 valid_f1_macro 由大到小排序)


,rank,resampling,feature_selection,n_features,train_cv3_f1_macro,valid_overall_acc,valid_boy_acc,valid_girl_acc,valid_f1_macro
0,1,none,l1_based,12,0.808826,0.917647,0.936508,0.863636,0.894222
1,2,none,top_30_rf,30,0.786709,0.917647,0.936508,0.863636,0.894222
2,3,xgb_cost_sensitive,all_features,41,0.798037,0.905882,0.904762,0.909091,0.883880
3,4,smote,top_15_rf,15,0.767770,0.905882,0.920635,0.863636,0.880785
4,5,smote,top_20_rf,20,0.773415,0.905882,0.920635,0.863636,0.880785
5,6,smote,top_30_rf,30,0.767525,0.905882,0.920635,0.863636,0.880785
6,7,smote,all_features,41,0.776959,0.905882,0.920635,0.863636,0.880785
7,8,none,rf_importance_mean_threshold,10,0.795405,0.905882,0.936508,0.818182,0.877345
8,9,none,all_features,41,0.782443,0.905882,0.936508,0.818182,0.877345
9,10,none,top_20_rf,20,0.783401,0.905882,0.936508,0.818182,0.877345



所有 No-CV 組合詳細性能對比 (按 valid_f1_macro 由大到小排序)


,rank,resampling,feature_selection,n_features,train_cv3_f1_macro,valid_overall_acc,valid_boy_acc,valid_girl_acc,valid_f1_macro
0,1,none,l1_based,12,NaN,0.917647,0.936508,0.863636,0.894222
1,2,none,top_30_rf,30,NaN,0.917647,0.936508,0.863636,0.894222
2,3,xgb_cost_sensitive,all_features,41,NaN,0.905882,0.904762,0.909091,0.883880
3,4,smote,top_15_rf,15,NaN,0.905882,0.920635,0.863636,0.880785
4,5,smote,top_20_rf,20,NaN,0.905882,0.920635,0.863636,0.880785
5,6,smote,top_30_rf,30,NaN,0.905882,0.920635,0.863636,0.880785
6,7,smote,all_features,41,NaN,0.905882,0.920635,0.863636,0.880785
7,8,none,rf_importance_mean_threshold,10,NaN,0.905882,0.936508,0.818182,0.877345
8,9,none,all_features,41,NaN,0.905882,0.936508,0.818182,0.877345
9,10,none,top_20_rf,20,NaN,0.905882,0.936508,0.818182,0.877345


## 產生 Kaggle Submission 檔案

In [17]:
# 產生 Kaggle Submission：支援 CV / No-CV + 指定特定輸出
# ---------- Export control ----------
# export_mode:
# - 'all'  : 匯出 CV top-k + No-CV top-k
# - 'cv'   : 只匯出 CV top-k
# - 'nocv' : 只匯出 No-CV top-k
# - 'single': 只匯出單一指定組合 (由 single_target 或 single_spec 控制)
export_mode = 'all'

# for mode in {'all', 'cv', 'nocv'}
export_topk = 3

# for mode == 'single': 依 rank 指定
# tag: 'cv' or 'nocv', rank 從 1 開始
single_target = {
    'tag': 'nocv',
    'rank': 1
}

# for mode == 'single': 也可改用策略指定（優先於 single_target）
# 若不使用，保留 None 即可
single_spec = None
# 範例：
# single_spec = {
#     'tag': 'cv',
#     'resampling': 'xgb_cost_sensitive',
#     'feature_selection': 'top_30_rf'
# }


if 'test_ids' not in globals() or test_ids is None:
    print('Error: test_ids 尚未建立，請先執行前處理 cell。')
else:
    def get_artifacts_by_tag(tag):
        if tag == 'cv':
            return top3_artifacts if 'top3_artifacts' in globals() else []
        if tag == 'nocv':
            return top3_artifacts_nocv if 'top3_artifacts_nocv' in globals() else []
        raise ValueError(f'Unknown tag: {tag}')

    def export_items(items, tag):
        if items is None or len(items) == 0:
            print(f'[{tag}] 無可用 artifacts，略過。')
            return []

        saved_files = []
        for i, art in enumerate(items, start=1):
            preds_cv = art['model'].predict(art['X_test_sel'])
            preds = preds_cv + label_offset

            submission_i = pd.DataFrame({
                'id': test_ids,
                'gender': preds
            })

            out_path = f'submission_{tag}_{i}_{art["resampling"]}_{art["feature_selection"]}.csv'
            submission_i.to_csv(out_path, index=False)
            saved_files.append(out_path)
            print(f'[{tag}] {out_path} 儲存成功')

        return saved_files

    def pick_by_rank(artifacts, rank_1based):
        if rank_1based < 1 or rank_1based > len(artifacts):
            raise ValueError(f'rank 超出範圍: {rank_1based}, 可用範圍 1..{len(artifacts)}')
        return [artifacts[rank_1based - 1]]

    def pick_by_spec(artifacts, resampling, feature_selection):
        picked = [
            a for a in artifacts
            if a.get('resampling') == resampling and a.get('feature_selection') == feature_selection
        ]
        if len(picked) == 0:
            raise ValueError(f'找不到指定組合: resampling={resampling}, feature_selection={feature_selection}')
        return [picked[0]]

    cv_saved, nocv_saved = [], []
   
    if export_mode == 'all':

        cv_items = get_artifacts_by_tag('cv')[:export_topk]
        nocv_items = get_artifacts_by_tag('nocv')[:export_topk]
        cv_saved = export_items(cv_items, 'cv')
        nocv_saved = export_items(nocv_items, 'nocv')

    elif export_mode in ['cv', 'nocv']:
        items = get_artifacts_by_tag(export_mode)[:export_topk]
        saved = export_items(items, export_mode)
        if export_mode == 'cv':
            cv_saved = saved
        else:
            nocv_saved = saved

    elif export_mode == 'single':
        if single_spec is not None:
            tag = single_spec['tag']
            artifacts = get_artifacts_by_tag(tag)
            items = pick_by_spec(
                artifacts,
                single_spec['resampling'],
                single_spec['feature_selection']
            )
            saved = export_items(items, tag)
        else:
            tag = single_target['tag']
            artifacts = get_artifacts_by_tag(tag)
            items = pick_by_rank(artifacts, single_target['rank'])
            saved = export_items(items, tag)

        if tag == 'cv':
            cv_saved = saved
        else:
            nocv_saved = saved

    else:
        raise ValueError("export_mode 必須是 'all'/'cv'/'nocv'/'single'")

    if len(cv_saved) == 0 and len(nocv_saved) == 0:
        if 'final_model' in globals() and 'X_test_best' in globals():
            test_preds_cv = final_model.predict(X_test_best)
            test_preds = test_preds_cv + label_offset
            submission = pd.DataFrame({'id': test_ids, 'gender': test_preds})
            fallback_path = 'submission_fallback.csv'
            submission.to_csv(fallback_path, index=False)
            print(f'fallback 輸出成功: {fallback_path}')
            print(submission.head())
        else:
            print('找不到可輸出的模型（top3_artifacts / top3_artifacts_nocv / final_model）。')
    else:
        if len(cv_saved) > 0:
            print('\nCV 首個輸出預覽：')
            print(pd.read_csv(cv_saved[0]).head())
        if len(nocv_saved) > 0:
            print('\nNo-CV 首個輸出預覽：')
            print(pd.read_csv(nocv_saved[0]).head())

[cv] submission_cv_1_none_l1_based.csv 儲存成功
[cv] submission_cv_2_none_top_30_rf.csv 儲存成功
[cv] submission_cv_3_xgb_cost_sensitive_all_features.csv 儲存成功
[nocv] submission_nocv_1_none_l1_based.csv 儲存成功
[nocv] submission_nocv_2_none_top_30_rf.csv 儲存成功
[nocv] submission_nocv_3_xgb_cost_sensitive_all_features.csv 儲存成功

CV 首個輸出預覽：
   id  gender
0   1       1
1   2       1
2   3       2
3   4       1
4   5       1

No-CV 首個輸出預覽：
   id  gender
0   1       1
1   2       1
2   3       2
3   4       1
4   5       1
